# DINOv3 validation on rebuilt central-target crops

This notebook performs **evaluation only**. It never calls the training pipeline, does not run backward, and does not modify the supplied DINO checkpoint. It rebuilds the retrieval index from the latest crop dataset, evaluates the previous `best.pt`, exports metrics, and visualizes two error groups: the worst ranks and cases where the ground truth is ranked second.

Before running, attach four Kaggle inputs: `viscaner-dinov3-data`, `viscaner-dinov3-vitb16-weights`, `viscaner-dinov3-code`, and the saved output of the previous training notebook containing `models/dinov3_retrieval/best.pt`.

## 1. Setup and copy the versioned pipeline

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

INPUT_ROOT = Path("/kaggle/input")
PROJECT_ROOT = Path("/kaggle/working/VIscaNEr")
CODE_INPUT = INPUT_ROOT / "viscaner-dinov3-code"
DATA_INPUT = INPUT_ROOT / "viscaner-dinov3-data"
WEIGHTS_INPUT = INPUT_ROOT / "viscaner-dinov3-vitb16-weights"

required_inputs = [CODE_INPUT, DATA_INPUT, WEIGHTS_INPUT]
for path in required_inputs:
    print(path, "OK" if path.exists() else "MISSING")
assert all(path.exists() for path in required_inputs), "Attach the three VIscaNEr Kaggle datasets first"

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
for source in CODE_INPUT.rglob("*"):
    if source.is_file() and source.name != "dataset-metadata.json":
        destination = PROJECT_ROOT / source.relative_to(CODE_INPUT)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers>=5.6,<6", "safetensors>=0.5,<1", "scikit-learn>=1.5,<2"],
    check=True,
)
print("Pipeline copied to", PROJECT_ROOT)

## 2. Resolve and validate every input

In [ ]:
checkpoint_candidates = sorted(
    [
        path for path in INPUT_ROOT.rglob("best.pt")
        if path.is_file() and path.stat().st_size > 100_000_000
    ],
    key=lambda path: path.stat().st_size,
    reverse=True,
)

print("DINO checkpoint candidates:")
for path in checkpoint_candidates:
    print(f"  {path} ({path.stat().st_size / 1024**2:.1f} MB)")
assert checkpoint_candidates, (
    "No trained DINO best.pt found. Attach the saved output of the previous training notebook."
)
OLD_BEST = checkpoint_candidates[0]

PATHS = {
    "checkpoint": OLD_BEST,
    "base_weights": WEIGHTS_INPUT / "model.safetensors",
    "metadata": DATA_INPUT / "crops_metadata.csv",
    "manifest": DATA_INPUT / "bottle_images_manifest.csv",
    "successful_crops": DATA_INPUT / "crops" / "successful",
    "low_confidence_crops": DATA_INPUT / "crops" / "low_confidence",
    "references": DATA_INPUT / "refs",
}
for name, path in PATHS.items():
    print(f"{name:24s} {"OK" if path.exists() else "MISSING"}  {path}")
assert all(path.exists() for path in PATHS.values())
print("Selected checkpoint:", OLD_BEST)

## 3. Run validation once

The command below calls only `validate`. Standard output and progress are written to one fresh log file opened in overwrite mode, so rerunning the cell does not append duplicate records. The CLI also emits the dataset summary only once in its final JSON.

In [ ]:
RUN_NAME = "dinov3_validation_new_crops"
RUN_DIR = PROJECT_ROOT / "runs" / "dinov3_retrieval" / RUN_NAME
LOG_PATH = RUN_DIR / "validation.log"
RUN_DIR.mkdir(parents=True, exist_ok=True)

command = [
    sys.executable,
    str(PROJECT_ROOT / "train_dinov3_retrieval.py"),
    "validate",
    "--config", str(PROJECT_ROOT / "configs" / "dinov3_retrieval.yaml"),
    "--project-root", str(PROJECT_ROOT),
    "--weights-path", str(PATHS["base_weights"]),
    "--crops-metadata-path", str(PATHS["metadata"]),
    "--bottle-manifest-path", str(PATHS["manifest"]),
    "--crops-root", str(DATA_INPUT / "crops"),
    "--refs-root", str(PATHS["references"]),
    "--checkpoint", str(OLD_BEST),
    "--run-name", RUN_NAME,
    "--device", "cuda",
    "--num-workers", "4",
    "--eval-batch-size", "64",
]

print("Starting evaluation-only run. Log:", LOG_PATH)
with LOG_PATH.open("w", encoding="utf-8") as log_file:
    completed = subprocess.run(
        command,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )
if completed.returncode != 0:
    print(LOG_PATH.read_text(encoding="utf-8")[-12000:])
    raise RuntimeError(f"Validation failed with exit code {completed.returncode}")
print("Validation completed successfully. Full log:", LOG_PATH)

## 4. Metrics and dataset coverage

In [ ]:
import pandas as pd
from IPython.display import display

SUMMARY_PATH = RUN_DIR / "validation_only_summary.json"
summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
metrics = summary["metrics_on_current_data"]
metrics_table = pd.DataFrame(metrics).T.reset_index(names="split")
display(metrics_table.style.format({
    "recall_at_1": "{:.2%}",
    "recall_at_5": "{:.2%}",
    "recall_at_10": "{:.2%}",
    "mrr": "{:.4f}",
    "median_rank": "{:.1f}",
    "mean_rank": "{:.1f}",
}))

data_summary = summary["current_data_summary"]
status_counts = data_summary["source_status_counts"]
total = sum(status_counts.values())
crop_coverage = (status_counts.get("successful", 0) + status_counts.get("low_confidence", 0)) / total
print(f"Crop coverage: {crop_coverage:.2%} ({total - status_counts.get('failed', 0):,}/{total:,})")
print("Status counts:", status_counts)
print("Leakage train vs unseen:", data_summary["leakage_check_train_vs_unseen"])

## 5. Error groups

In [ ]:
audit = summary["audit"]
category_table = pd.DataFrame(audit["category_counts"]).T.reset_index(names="split")
display(category_table)
print("Full per-query audit CSV:", audit["audit_csv"])

In [ ]:
from PIL import Image

preferred_order = [
    "val_seen_worst", "val_seen_rank2",
    "val_unseen_worst", "val_unseen_rank2",
    "val_hard_worst", "val_hard_rank2",
]
for key in preferred_order:
    image_path = audit["images"].get(key)
    if image_path:
        print("\n", key, "→", image_path)
        display(Image.open(image_path))

## 6. Inspect rank-2 cases as a table

In [ ]:
audit_df = pd.read_csv(audit["audit_csv"])
rank2_cases = audit_df[audit_df["rank"].eq(2)].copy()
rank2_cases["wrong_lead"] = rank2_cases["top1_similarity"] - rank2_cases["true_similarity"]
display(
    rank2_cases.sort_values("wrong_lead", ascending=False)[
        ["split", "true_slug", "top1_slug", "true_similarity",
         "top1_similarity", "wrong_lead", "query_path"]
    ].head(50)
)

## Outputs

- `validation_only_metrics.json` — metrics on the rebuilt crops.
- `validation_only_summary.json` — paths, old checkpoint metrics and current metrics.
- `validation_only_audit/retrieval_audit.csv` — every query and its Top-5 results.
- `validation_only_audit/*_worst.png` — worst retrieval mistakes.
- `validation_only_audit/*_rank2.png` — mistakes where the correct product is Top-2.
- `validation.log` — exactly one overwritten log for the run.